# dbt Data Contracts & Model Versioning

A live demo using **dbt-core + DuckDB**. No warehouse required — everything runs locally.

---

## What we'll cover

| Chapter | Topic |
|---------|-------|
| 1 | Raw data — seed tables as the source of truth |
| 2 | Staging layer — transformations without contracts |
| 3 | Data contracts — locking the gold layer |
| 4 | **Contract violation** — what dbt catches and why it matters |
| 5 | **Model versioning** — evolving contracts without breaking consumers |

In [1]:
import subprocess
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_DIR = Path.cwd()
DB_PATH = PROJECT_DIR / "demo.duckdb"

print(f"Project : {PROJECT_DIR}")
print(f"Database: {DB_PATH}")

Project : /Users/NoeFischer/gh_nf/dbt-contracts-demo
Database: /Users/NoeFischer/gh_nf/dbt-contracts-demo/demo.duckdb


In [2]:
import shutil

# ── Clean slate ────────────────────────────────────────────────────────────
# Remove all dbt artifacts and the database so every run starts fresh.
artifacts = [
    PROJECT_DIR / "demo.duckdb",
    PROJECT_DIR / "demo.duckdb.wal",
    PROJECT_DIR / "target",
    PROJECT_DIR / "logs",
    PROJECT_DIR / "dbt_packages",
]

removed = []
for path in artifacts:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
        removed.append(path.name)

if removed:
    print(f"Removed: {', '.join(removed)}")
else:
    print("Nothing to clean.")
print("Clean slate ready ✓")

Removed: demo.duckdb, target, logs
Clean slate ready ✓


## Setup: Build the dbt Pipeline

All artifacts from previous runs (`demo.duckdb`, `target/`, `logs/`) have been removed. Now load the seed data and build all models from scratch.

In [3]:
def run_dbt(args):
    """Run a dbt command, print concise output, return the result."""
    cmd = ["dbt"] + args + ["--profiles-dir", str(PROJECT_DIR)]
    print(f"$ dbt {' '.join(args)}")
    print("─" * 70)
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_DIR))

    if result.returncode == 0:
        for line in result.stdout.splitlines():
            # Show: model counts, per-model OK lines, timing summary, done
            if any(kw in line for kw in ["Found", " OK ", "Finished running", "Done."]):
                print(line)
        print("\n✅  Completed successfully")
    else:
        print(result.stdout)
        if result.stderr.strip():
            print(result.stderr)

    return result


# Step 1: load CSVs into main_raw schema
run_dbt(["seed"])
print()

# Step 2: build all models (skip the intentionally broken one — that comes in Chapter 4)
run_dbt(["run", "--exclude", "fct_orders_broken"])

$ dbt seed
──────────────────────────────────────────────────────────────────────


21:26:56  Found 9 models, 3 seeds, 474 macros
21:26:56  1 of 3 OK loaded seed file main_raw.raw_customers .............................. [INSERT 6 in 0.04s]
21:26:56  2 of 3 OK loaded seed file main_raw.raw_orders ................................. [INSERT 20 in 0.01s]
21:26:56  3 of 3 OK loaded seed file main_raw.raw_products ............................... [INSERT 5 in 0.01s]
21:26:56  Finished running 3 seeds in 0 hours 0 minutes and 0.14 seconds (0.14s).
21:26:56  Done. PASS=3 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=3

✅  Completed successfully

$ dbt run --exclude fct_orders_broken
──────────────────────────────────────────────────────────────────────


21:26:58  Running with dbt=1.11.7
21:26:58  Registered adapter: duckdb=1.10.1
21:26:58  Found 9 models, 3 seeds, 474 macros
21:26:58  The selection criterion 'fct_orders_broken' does not match any enabled nodes
21:26:58  
21:26:58  Concurrency: 1 threads (target='dev')
21:26:58  
21:26:58  1 of 9 START sql view model main_staging.stg_customers ......................... [RUN]
21:26:58  1 of 9 OK created sql view model main_staging.stg_customers .................... [OK in 0.04s]
21:26:58  2 of 9 START sql view model main_staging.stg_orders ............................ [RUN]
21:26:58  2 of 9 OK created sql view model main_staging.stg_orders ....................... [OK in 0.01s]
21:26:58  3 of 9 START sql view model main_staging.stg_products .......................... [RUN]
21:26:58  3 of 9 OK created sql view model main_staging.stg_products ..................... [OK in 0.01s]
21:26:58  4 of 9 START sql table model main_gold.gold_dim_customers ...................... [RUN]
21:26:58  4 of 9

CompletedProcess(args=['dbt', 'run', '--exclude', 'fct_orders_broken', '--profiles-dir', '/Users/NoeFischer/gh_nf/dbt-contracts-demo'], returncode=1, stdout="\x1b21:26:58  Running with dbt=1.11.7\n\x1b21:26:58  Registered adapter: duckdb=1.10.1\n\x1b21:26:58  Found 9 models, 3 seeds, 474 macros\n\x1b21:26:58  The selection criterion 'fct_orders_broken' does not match any enabled nodes\n\x1b21:26:58  \n\x1b21:26:58  Concurrency: 1 threads (target='dev')\n\x1b21:26:58  \n\x1b21:26:58  1 of 9 START sql view model main_staging.stg_customers ......................... [RUN]\n\x1b21:26:58  1 of 9 OK created sql view model main_staging.stg_customers .................... [\x1bOK\x1b in 0.04s]\n\x1b21:26:58  2 of 9 START sql view model main_staging.stg_orders ............................ [RUN]\n\x1b21:26:58  2 of 9 OK created sql view model main_staging.stg_orders ....................... [\x1bOK\x1b in 0.01s]\n\x1b21:26:58  3 of 9 START sql view model main_staging.stg_products ..................

In [4]:
# Open a connection to the database (dbt has already written everything)
con = duckdb.connect(str(DB_PATH))

print("Connected to demo.duckdb ✓\n")
print("All tables in the database:")
con.sql("SHOW ALL TABLES").df()[["database", "schema", "name", "column_names"]]

Connected to demo.duckdb ✓

All tables in the database:


,database,schema,name,column_names
0,demo,main_gold,gold_dim_customers,"[customer_id, name, email, country]"
1,demo,main_gold,gold_fct_orders_v1,"[order_id, customer_id, product_id, status, or..."
2,demo,main_gold,gold_fct_orders_v2,"[order_id, customer_id, product_id, status, or..."
3,demo,main_raw,raw_customers,"[customer_id, name, email, country]"
4,demo,main_raw,raw_orders,"[order_id, customer_id, product_id, amount, st..."
5,demo,main_raw,raw_products,"[product_id, name, category, price]"
6,demo,main_serving,dim_customers,"[customer_id, name, email, country]"
7,demo,main_serving,fct_orders,"[order_id, customer_id, product_id, status, or..."
8,demo,main_staging,stg_customers,"[customer_id, name, email, country]"
9,demo,main_staging,stg_orders,"[order_id, customer_id, product_id, amount, st..."


## Pipeline Architecture

```
┌──────────────────┐   ┌───────────────────────┐   ┌───────────────────────┐   ┌──────────────────────┐
│   main_raw       │   │   main_staging        │   │   main_gold           │   │   main_serving       │
│──────────────────│   │───────────────────────│   │───────────────────────│   │──────────────────────│
│ raw_orders       │──▶│ stg_orders  (view)    │──▶│ fct_orders_v1  table  │   │ fct_orders  (view)   │
│ raw_customers    │──▶│ stg_customers (view)  │──▶│ fct_orders_v2  table  │──▶│ dim_customers (view) │
│ raw_products     │──▶│ stg_products  (view)  │──▶│ dim_customers  table  │   │                      │
│                  │   │                       │   │                       │   │                      │
│ Seeds (CSV)      │   │ No contracts          │   │ ✓ Contracts enforced  │   │ Public interface     │
└──────────────────┘   └───────────────────────┘   └───────────────────────┘   └──────────────────────┘
```

**Contracts lock the gold layer** — stable schema guarantee for all consumers.
**The serving layer** exposes versioned views: consumers query `fct_orders` and `dim_customers`, never the gold tables directly.

---
## Chapter 1: Raw Data

Three seed tables loaded from CSV files. This is our source of truth — no transformations, no guarantees, just raw records.

In [5]:
for table, label in [
    ("main_raw.raw_customers", "raw_customers — 6 rows"),
    ("main_raw.raw_products",  "raw_products — 5 rows"),
    ("main_raw.raw_orders",    "raw_orders — 20 rows"),
]:
    print(f"\n{'━' * 60}")
    print(f"  {label}")
    print(f"{'━' * 60}")
    display(con.sql(f"SELECT * FROM {table}").df())


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  raw_customers — 6 rows
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,customer_id,name,email,country
0,101,Alice Martin,alice@example.com,US
1,102,Bob Chen,bob@example.com,CA
2,103,Clara Sanz,clara@example.com,ES
3,104,David Kim,david@example.com,KR
4,105,Eva Brown,eva@example.com,US
5,106,Frank Liu,frank@example.com,CN



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  raw_products — 5 rows
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,product_id,name,category,price
0,201,Wireless Earbuds,Electronics,49.99
1,202,Laptop Stand,Accessories,129.00
2,203,USB-C Cable,Accessories,19.99
3,204,Mechanical Keyboard,Electronics,249.99
4,205,Desk Lamp,Home,89.99



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  raw_orders — 20 rows
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,order_id,customer_id,product_id,amount,status,created_at
0,1,101,201,49.99,completed,2024-01-15
1,2,102,202,129.00,completed,2024-01-16
2,3,101,203,19.99,pending,2024-01-17
3,4,103,201,49.99,completed,2024-01-18
4,5,104,204,249.99,shipped,2024-01-19
5,6,102,203,19.99,cancelled,2024-01-20
6,7,105,202,129.00,completed,2024-01-21
7,8,103,205,89.99,pending,2024-01-22
8,9,106,201,49.99,completed,2024-01-23
9,10,101,204,249.99,shipped,2024-01-24


---
## Chapter 2: The Staging Layer — Transformations, No Contracts

Staging models cast raw types, rename columns, and clean the data. They are **intentionally uncontracted** — this is the fast-iteration zone.

```sql
-- models/staging/stg_orders.sql
select
    cast(order_id    as bigint)        as order_id,
    cast(customer_id as bigint)        as customer_id,
    cast(product_id  as bigint)        as product_id,
    cast(amount      as decimal(10,2)) as amount,
    cast(status      as varchar)       as status,
    cast(created_at  as date)          as created_at
from {{ ref('raw_orders') }}
```

No `contract: enforced: true` here. If a developer renames `order_id` to `id` in this model, dbt will happily run it. The problem only surfaces downstream.

In [6]:
print("stg_orders — column types (no contract, no constraints):")
display(con.sql("DESCRIBE main_staging.stg_orders").df())

print("\nDatabase-level constraints on stg_orders:")
constraints = con.sql("""
    SELECT constraint_type, constraint_column_names
    FROM duckdb_constraints()
    WHERE table_name = 'stg_orders'
""").df()
if constraints.empty:
    print("  (none — staging is a view with no enforced constraints)")
else:
    display(constraints)

print("\nFirst 5 rows:")
display(con.sql("SELECT * FROM main_staging.stg_orders LIMIT 5").df())

stg_orders — column types (no contract, no constraints):


,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,customer_id,BIGINT,YES,None,None,None
2,product_id,BIGINT,YES,None,None,None
3,amount,"DECIMAL(10,2)",YES,None,None,None
4,status,VARCHAR,YES,None,None,None
5,created_at,DATE,YES,None,None,None



Database-level constraints on stg_orders:
  (none — staging is a view with no enforced constraints)

First 5 rows:


,order_id,customer_id,product_id,amount,status,created_at
0,1,101,201,49.99,completed,2024-01-15
1,2,102,202,129.00,completed,2024-01-16
2,3,101,203,19.99,pending,2024-01-17
3,4,103,201,49.99,completed,2024-01-18
4,5,104,204,249.99,shipped,2024-01-19


---
## Chapter 3: Data Contracts — Locking the Gold Layer

At the gold layer, models declare a **data contract**: an explicit promise about column names, types, and constraints.

```yaml
# models/gold/_dim_customers.yml
models:
  - name: gold_dim_customers
    config:
      contract:
        enforced: true          # ← turns on contract enforcement
    columns:
      - name: customer_id
        data_type: bigint       # ← declared type
        constraints:
          - type: not_null      # ← real DB-level NOT NULL
          - type: unique        # ← real DB-level UNIQUE
      - name: name
        data_type: varchar
      - name: email
        data_type: varchar
      - name: country
        data_type: varchar
```

When `enforced: true`, dbt generates a `CREATE TABLE` with **explicit column types and constraints**, then inserts the model's output. If the output doesn't match the declared schema → **build fails immediately**.

In [7]:
print("dim_customers — column schema (contract enforced):")
display(con.sql("DESCRIBE main_gold.gold_dim_customers").df())

print("\nDatabase-level constraints applied to dim_customers:")
display(
    con.sql("""
        SELECT constraint_type, constraint_column_names
        FROM duckdb_constraints()
        WHERE table_name = 'gold_dim_customers'
    """).df()
)

dim_customers — column schema (contract enforced):


,column_name,column_type,null,key,default,extra
0,customer_id,BIGINT,NO,UNI,None,None
1,name,VARCHAR,YES,NaN,None,None
2,email,VARCHAR,YES,NaN,None,None
3,country,VARCHAR,YES,NaN,None,None



Database-level constraints applied to dim_customers:


,constraint_type,constraint_column_names
0,NOT NULL,[customer_id]
1,UNIQUE,[customer_id]


In [8]:
print("fct_orders_v2 — column schema (contract enforced):")
display(con.sql("DESCRIBE main_gold.gold_fct_orders_v2").df())

print("\nDatabase-level constraints applied to fct_orders_v2:")
display(
    con.sql("""
        SELECT constraint_type, constraint_column_names
        FROM duckdb_constraints()
        WHERE table_name = 'gold_fct_orders_v2'
    """).df()
)

fct_orders_v2 — column schema (contract enforced):


,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,NO,PRI,None,None
1,customer_id,BIGINT,NO,NaN,None,None
2,product_id,BIGINT,NO,NaN,None,None
3,status,VARCHAR,YES,NaN,None,None
4,order_date,DATE,YES,NaN,None,None
5,order_total,"DECIMAL(10,2)",NO,NaN,None,None
6,product_category,VARCHAR,YES,NaN,None,None



Database-level constraints applied to fct_orders_v2:


,constraint_type,constraint_column_names
0,NOT NULL,[order_id]
1,NOT NULL,[customer_id]
2,NOT NULL,[product_id]
3,NOT NULL,[order_total]
4,PRIMARY KEY,[order_id]


In [9]:
print("dim_customers data (6 rows):")
display(con.sql("SELECT * FROM main_gold.gold_dim_customers").df())

print("\nfct_orders_v2 data (first 5 rows):")
display(con.sql("SELECT * FROM main_gold.gold_fct_orders_v2 LIMIT 5").df())

dim_customers data (6 rows):


,customer_id,name,email,country
0,101,Alice Martin,alice@example.com,US
1,102,Bob Chen,bob@example.com,CA
2,103,Clara Sanz,clara@example.com,ES
3,104,David Kim,david@example.com,KR
4,105,Eva Brown,eva@example.com,US
5,106,Frank Liu,frank@example.com,CN



fct_orders_v2 data (first 5 rows):


,order_id,customer_id,product_id,status,order_date,order_total,product_category
0,1,101,201,completed,2024-01-15,49.99,Electronics
1,2,102,202,completed,2024-01-16,129.00,Accessories
2,3,101,203,pending,2024-01-17,19.99,Accessories
3,4,103,201,completed,2024-01-18,49.99,Electronics
4,5,104,204,shipped,2024-01-19,249.99,Electronics


### Under the hood: the generated `CREATE TABLE`

When `contract: enforced: true`, dbt generates a `CREATE TABLE` statement with **explicit types and constraints** baked in, then inserts the model's output into it. Here's the actual DDL dbt compiled for `fct_orders_v2`:

In [10]:
import re

ddl_path = PROJECT_DIR / "target/run/dbt_contracts_demo/models/gold/gold_fct_orders_v2.sql"
ddl_text = ddl_path.read_text()

# Extract just the CREATE TABLE...;  block (everything up to the first semicolon)
start = ddl_text.lower().find("create")
end = ddl_text.find(";", start) + 1
create_stmt = ddl_text[start:end]

# Clean up Jinja whitespace artifacts
create_stmt = re.sub(r"[ \t]+\n", "\n", create_stmt)
create_stmt = re.sub(r"\n{3,}", "\n\n", create_stmt)
create_stmt = re.sub(r"  +", " ", create_stmt)
create_stmt = create_stmt.strip()

print(create_stmt)
print()
print("─" * 70)
print("Every column type and constraint comes directly from the YAML contract.")
print("dbt generated this DDL — the developer never writes it by hand.")

create table
 "demo"."main_gold"."gold_fct_orders_v2__dbt_tmp"

 (
 order_id bigint not null,
 customer_id bigint not null,
 product_id bigint not null,
 status varchar,
 order_date date,
 order_total decimal(10,2) not null,
 product_category varchar,

 primary key (order_id)
 )
 ;

──────────────────────────────────────────────────────────────────────
Every column type and constraint comes directly from the YAML contract.
dbt generated this DDL — the developer never writes it by hand.


---
## Chapter 4: Contract Violation

A developer renames `order_id` → `id` in `fct_orders`. The SQL is valid. Without contracts, this deploys silently and breaks every downstream consumer.

```sql
-- models/serving/fct_orders_broken.sql
select
    o.order_id  as id,          -- renamed: contract expects 'order_id'
    o.customer_id,
    o.product_id,
    o.amount,
    o.status,
    o.created_at    as order_date,
    p.category      as product_category
from orders o
left join products p on o.product_id = p.product_id
```

The contract declares `order_id bigint not null`. Let's run it.

In [11]:
print("Running the broken model against its contract...\n")

# DuckDB allows only one writer at a time.
# We close our read connection so dbt can acquire the write lock.
con.close()

cmd = [
    "dbt", "run",
    "--select", "fct_orders_broken",
    "--profiles-dir", str(PROJECT_DIR),
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_DIR))

print(result.stdout)
if result.stderr.strip():
    print(result.stderr)

if result.returncode != 0:
    print("\n" + "═" * 70)
    print("  ❌  dbt REFUSED to build this model")
    print("═" * 70)

# Reopen for subsequent analysis cells
con = duckdb.connect(str(DB_PATH))

Running the broken model against its contract...



21:27:00  Running with dbt=1.11.7
21:27:00  Registered adapter: duckdb=1.10.1
21:27:00  Found 9 models, 3 seeds, 474 macros
21:27:00  The selection criterion 'fct_orders_broken' does not match any enabled nodes
21:27:00  Nothing to do. Try checking your model configs and model specification args



### What dbt caught

dbt shows exactly what differs between the model output and the contract:

```
| column_name | definition_type | contract_type | mismatch_reason       |
| ----------- | --------------- | ------------- | --------------------- |
| id          | BIGINT          |               | missing in contract   |
| order_id    |                 | BIGINT        | missing in definition |
```

**Nothing was written to the database.** The build failed at compile time — caught before any SQL ran, not at 2 am.

---
## Chapter 5: Model Versioning — Evolving Contracts Without Breaking Consumers

Contracts prevent silent breakage, but what if you *need* to change the schema? Adding `product_category` is backward-incompatible. The answer: **model versioning**.

Both versions are built simultaneously as separate gold tables. The serving view controls which version consumers get:

```yaml
# models/gold/_fct_orders.yml
models:
  - name: gold_fct_orders_v1   # 6 columns — original, stable
    config:
      contract:
        enforced: true
    columns:
      - name: order_id
        data_type: bigint
        constraints: [not_null]
      # ... (customer_id, product_id, amount, status, order_date)

  - name: gold_fct_orders_v2   # 7 columns — adds product_category
    config:
      contract:
        enforced: true
    columns:
      - include: "*"  # same base columns
      - name: product_category
        data_type: varchar
```

```sql
-- models/serving/fct_orders.sql  ← the version resolver
select * from {{ ref('gold_fct_orders_v2') }}
```

To cut over: change one line in `fct_orders.sql`. Consumers always query `main_serving.fct_orders` — no code changes on their end.

In [12]:
v1 = con.sql("SELECT * FROM main_gold.gold_fct_orders_v1 LIMIT 5").df()
v2 = con.sql("SELECT * FROM main_gold.gold_fct_orders_v2 LIMIT 5").df()

print(f"fct_orders  v1  — {len(v1.columns)} columns: {list(v1.columns)}")
display(v1)

print(f"\nfct_orders  v2  — {len(v2.columns)} columns: {list(v2.columns)}")
display(v2)

new_cols = [c for c in v2.columns if c not in v1.columns]
print(f"\nNew in v2: {new_cols}")

fct_orders  v1  — 6 columns: ['order_id', 'customer_id', 'product_id', 'status', 'order_date', 'amount']


,order_id,customer_id,product_id,status,order_date,amount
0,1,101,201,completed,2024-01-15,49.99
1,2,102,202,completed,2024-01-16,129.00
2,3,101,203,pending,2024-01-17,19.99
3,4,103,201,completed,2024-01-18,49.99
4,5,104,204,shipped,2024-01-19,249.99



fct_orders  v2  — 7 columns: ['order_id', 'customer_id', 'product_id', 'status', 'order_date', 'order_total', 'product_category']


,order_id,customer_id,product_id,status,order_date,order_total,product_category
0,1,101,201,completed,2024-01-15,49.99,Electronics
1,2,102,202,completed,2024-01-16,129.00,Accessories
2,3,101,203,pending,2024-01-17,19.99,Accessories
3,4,103,201,completed,2024-01-18,49.99,Electronics
4,5,104,204,shipped,2024-01-19,249.99,Electronics



New in v2: ['order_total', 'product_category']


In [13]:
# Gold layer: both versioned tables coexist
rows = []
for tbl, version, note in [
    ("gold_fct_orders_v1", "v1", "baseline — 6 cols, stable"),
    ("gold_fct_orders_v2", "v2 (latest)", "extended — 7 cols, adds product_category"),
    ("gold_dim_customers",  "—", "dimension table"),
]:
    row_count = con.sql(f"SELECT COUNT(*) FROM main_gold.{tbl}").fetchone()[0]
    cols = con.sql(
        f"SELECT column_name FROM information_schema.columns "
        f"WHERE table_name = '{tbl}' AND table_schema = 'main_gold' ORDER BY ordinal_position"
    ).df()["column_name"].tolist()
    rows.append({
        "table": f"main_gold.{tbl}",
        "version": version,
        "rows": row_count,
        "columns": len(cols),
        "column names": ", ".join(cols),
    })

display(pd.DataFrame(rows))

,table,version,rows,columns,column names
0,main_gold.gold_fct_orders_v1,v1,20,6,"order_id, customer_id, product_id, status, ord..."
1,main_gold.gold_fct_orders_v2,v2 (latest),20,7,"order_id, customer_id, product_id, status, ord..."
2,main_gold.gold_dim_customers,—,6,4,"customer_id, name, email, country"


In [14]:
# The serving layer — what consumers actually query
print("main_serving.fct_orders (view → gold_fct_orders_v2):")
display(con.sql("SELECT * FROM main_serving.fct_orders LIMIT 5").df())

print("\nmain_serving.dim_customers:")
display(con.sql("SELECT * FROM main_serving.dim_customers").df())

print("\nView definitions:")
display(con.sql("""
    SELECT table_name AS view_name, view_definition
    FROM information_schema.views
    WHERE table_schema = 'main_serving'
    ORDER BY table_name
""").df())

main_serving.fct_orders (view → gold_fct_orders_v2):


,order_id,customer_id,product_id,status,order_date,order_total,product_category
0,1,101,201,completed,2024-01-15,49.99,Electronics
1,2,102,202,completed,2024-01-16,129.00,Accessories
2,3,101,203,pending,2024-01-17,19.99,Accessories
3,4,103,201,completed,2024-01-18,49.99,Electronics
4,5,104,204,shipped,2024-01-19,249.99,Electronics



main_serving.dim_customers:


,customer_id,name,email,country
0,101,Alice Martin,alice@example.com,US
1,102,Bob Chen,bob@example.com,CA
2,103,Clara Sanz,clara@example.com,ES
3,104,David Kim,david@example.com,KR
4,105,Eva Brown,eva@example.com,US
5,106,Frank Liu,frank@example.com,CN



View definitions:


,view_name,view_definition
0,dim_customers,CREATE VIEW main_serving.dim_customers AS SELE...
1,fct_orders,CREATE VIEW main_serving.fct_orders AS SELECT ...


### Migration path

The cutover is a one-line change in `models/serving/fct_orders.sql`.

**Before cutover** — serving view points to v1:

| Layer | Model | Type | Note |
|---|---|---|---|
| `main_gold` | `gold_fct_orders_v1` | Table | Production |
| `main_gold` | `gold_fct_orders_v2` | Table | QA / staging |
| `main_serving` | `fct_orders` | **View** | `→ ref('gold_fct_orders_v1')` |

**After cutover** — update the ref; v1 gets a deprecation date:

| Layer | Model | Type | Note |
|---|---|---|---|
| `main_gold` | `gold_fct_orders_v1` | Table | Deprecated — grace period |
| `main_gold` | `gold_fct_orders_v2` | Table | Production |
| `main_serving` | `fct_orders` | **View** | `→ ref('gold_fct_orders_v2')` ✓ |

Consumers always query `main_serving.fct_orders` — they never need to change their code.

---
## Summary

| Feature | Key takeaway |
|---------|-------------|
| **`contract: enforced: true`** | Generates a typed `CREATE TABLE` — column names, types, and constraints declared in YAML, enforced by the DB |
| **Column constraints** | `not_null`, `unique`, `primary_key` at the database level — not just dbt tests |
| **Contract violation** | Build fails **before any data is written** if model output doesn't match the declared schema |
| **`latest_version: N`** | `ref('fct_orders')` auto-resolves to the current latest version |
| **Version pinning** | `ref('fct_orders', version=1)` — legacy consumers never break during a schema migration |

---

*Stack: dbt-core · dbt-duckdb · DuckDB*